**Homework - 6 (trees)**

In [1]:
#importing libraries
import sys
 
import numpy as np
import pandas as pd
import xgboost as xgb
 
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_extraction import DictVectorizer
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
 
 
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [9]:
# Preparing the dataset

# Works as a script and in Jupyter (Jupyter passes "-f kernel.json" in sys.argv).
csv_args = [a for a in sys.argv[1:] if a.endswith('.csv')]
path = csv_args[0] if csv_args else 'car_fuel_efficiency_2026.csv'
df = pd.read_csv(path)
 
print('Missing values per column:')
print(df.isnull().sum())
 
df = df.fillna(0)
 
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)
 
y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values
 
del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
 
dv = DictVectorizer(sparse=True)
X_train = dv.fit_transform(df_train.to_dict(orient='records'))
X_val = dv.transform(df_val.to_dict(orient='records'))
feature_names = list(dv.get_feature_names_out())

Missing values per column:
model_year               0
origin                   0
fuel_type                0
drivetrain               0
num_doors                0
engine_displacement      0
num_cylinders            0
horsepower             877
vehicle_weight           0
acceleration           264
fuel_efficiency_mpg      0
dtype: int64


In [14]:
# Question 1 : decision stump (max_depth=1) - which feature splits?..

dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)
 
split_feature = feature_names[dt.tree_.feature[0]]
print('\nQ1: root split on', split_feature,
      '-> original column.:', split_feature.split('=')[0])


Q1: root split on model_year -> original column.: model_year


In [16]:
# Question 2: random forest., n_estimators=10 -> validation RMSE.

rf = RandomForestRegressor(n_estimators=10, random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
print('\nQ2: RMSE =', round(rmse(y_val, rf.predict(X_val)), 3))


Q2: RMSE = 1.833


In [23]:
# Question 3: n_estimators from [10, 50, 100, 150]

print('\nQ3: RMSE per n_estimators')
q3 = {}
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(n_estimators=n, random_state=1, n_jobs=-1)
    rf.fit(X_train, y_train)
    q3[n] = round(rmse(y_val, rf.predict(X_val)), 3)
    print(f'n_estimators={n}: {q3[n]}')
best_val = min(q3.values())
print('Lowest RMSE at n_estimators =', min(n for n, r in q3.items() if r == best_val))


Q3: RMSE per n_estimators
n_estimators=10: 1.833
n_estimators=50: 1.774
n_estimators=100: 1.77
n_estimators=150: 1.769
Lowest RMSE at n_estimators = 150


In [32]:
# Question 4: max_depth x n_estimators, mean RMSE per max_depth.

print('\nQ4: mean RMSE per max_depth')
q4 = {}
for d in [10, 15, 20, 25]:
    scores = []
    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(n_estimators=n, max_depth=d,
                                   random_state=1, n_jobs=-1)
        rf.fit(X_train, y_train)
        scores.append(rmse(y_val, rf.predict(X_val)))
    q4[d] = np.mean(scores)
    print(f'max_depth={d}: mean RMSE = {q4[d]:.4f}')
print('Best max_depth:', min(q4, key=q4.get))


Q4: mean RMSE per max_depth
max_depth=10: mean RMSE = 1.7544
max_depth=15: mean RMSE = 1.7842
max_depth=20: mean RMSE = 1.7868
max_depth=25: mean RMSE = 1.7863
Best max_depth: 10


In [37]:
# Question 5: feature importance.

rf = RandomForestRegressor(n_estimators=10, max_depth=20,
                           random_state=1, n_jobs=-1)
rf.fit(X_train, y_train)
 
importances = pd.Series(rf.feature_importances_, index=feature_names)
print('\nQ5: top feature importances')
print(importances.sort_values(ascending=False).head(6).round(4))
 
options = ['vehicle_weight', 'horsepower', 'acceleration', 'engine_displacement']
print('Most important among the options:', importances[options].idxmax())


Q5: top feature importances
model_year             0.2961
vehicle_weight         0.2033
fuel_type=Gasoline     0.1607
horsepower             0.0783
engine_displacement    0.0613
acceleration           0.0550
dtype: float64
Most important among the options: vehicle_weight


In [45]:
# Question 6: XGBoost, eta 0.3 vs 0.1

dtrain = xgb.DMatrix(X_train, label=y_train, feature_names=feature_names)
dval = xgb.DMatrix(X_val, label=y_val, feature_names=feature_names)
watchlist = [(dtrain, 'train'), (dval, 'val')]
 
print('\nQ6: XGBoost validation RMSE')
q6 = {}
for eta in [0.3, 0.1]:
    xgb_params = {
        'eta': eta,
        'max_depth': 6,
        'min_child_weight': 1,
 
        'objective': 'reg:squarederror',
        'nthread': 8,
 
        'seed': 1,
        'verbosity': 1,
    }
    evals_result = {}
    xgb.train(xgb_params, dtrain, num_boost_round=100,
              evals=watchlist, evals_result=evals_result, verbose_eval=False)
 
    val_rmse = evals_result['val']['rmse']
    q6[eta] = (val_rmse[-1], min(val_rmse))
    print(f'eta={eta}: final round = {val_rmse[-1]:.4f}, '
          f'best round = {min(val_rmse):.4f} (round {int(np.argmin(val_rmse)) + 1})')
 
final = {eta: round(v[0], 3) for eta, v in q6.items()}
if final[0.3] == final[0.1]:
    print('Both give equal value')
else:
    print('Best eta (final round):', min(final, key=final.get))


Q6: XGBoost validation RMSE
eta=0.3: final round = 1.8265, best round = 1.7320 (round 11)
eta=0.1: final round = 1.7248, best round = 1.7122 (round 40)
Best eta (final round): 0.1
